In [45]:
import pandas as pd
import numpy as np
import json
import os
import re



"""
location信息
"""

location_df = pd.read_csv('Location.csv')
location_df.rename(columns={'Geneid':'Gene'}, inplace=True)

location_df

,Gene,Chr,Start,End,Strand,Length
0,MYCTH_2114025,NC_016472.1,1381,2142,+,762
1,MYCTH_2293935,NC_016472.1,2744,3343,-,600
2,MYCTH_2293936,NC_016472.1,3344,3817,-,474
3,MYCTH_2051335,NC_016472.1,5330,6385,-,1056
4,MYCTH_2121898,NC_016472.1,15458,25533,+,10076
...,...,...,...,...,...,...
9287,MYCTH_103797,NC_016478.1,4089438,4091337,+,1900
9288,MYCTH_2071098,NC_016478.1,4095568,4097284,+,1717
9289,MYCTH_2114022,NC_016478.1,4098122,4098907,-,786
9290,MYCTH_2114023,NC_016478.1,4100606,4101293,-,688


In [46]:
"""
NCBI_gene_id信息
"""

ncbi_gene_id_df = pd.read_csv('NCBI_gene_id.csv', names=['NCBI_gene_id', 'Gene'])
ncbi_gene_id_df['Gene'] = ncbi_gene_id_df['Gene'].str.replace('mtm:','')
ncbi_gene_id_df['NCBI_gene_id'] = ncbi_gene_id_df['NCBI_gene_id'].str.replace('ncbi-geneid:','')

ncbi_gene_id_df

,NCBI_gene_id,Gene
0,11505753,MYCTH_100011
1,11512455,MYCTH_100068
2,11512426,MYCTH_100089
3,11512377,MYCTH_100127
4,11512195,MYCTH_100290
...,...,...
9286,11507534,MYCTH_t95
9287,11507441,MYCTH_t96
9288,11512736,MYCTH_t97
9289,11512532,MYCTH_t98


In [47]:
"""
NCBI protein id 信息
"""

protein_id_df = pd.read_csv('Protein_id.csv', names=['Protein_id', 'Gene'])
protein_id_df['Gene'] = protein_id_df['Gene'].str.replace('mtm:','')
protein_id_df['Protein_id'] = protein_id_df['Protein_id'].str.replace('ncbi-proteinid:','')

protein_id_df

,Protein_id,Gene
0,XP_003660026.1,MYCTH_100011
1,XP_003662402.1,MYCTH_100068
2,XP_003662373.1,MYCTH_100089
3,XP_003662324.1,MYCTH_100127
4,XP_003662142.1,MYCTH_100290
...,...,...
9092,XP_003659757.1,MYCTH_99786
9093,XP_003659791.1,MYCTH_99814
9094,XP_003659824.1,MYCTH_99838
9095,XP_003659843.1,MYCTH_99853


In [48]:
"""
uniport id 信息
"""

uniprot_id_df = pd.read_csv('Uniprot_id.csv', names=['Uniport_id', 'Gene'])
uniprot_id_df['Gene'] = uniprot_id_df['Gene'].str.replace('mtm:','')
uniprot_id_df['Uniport_id'] = uniprot_id_df['Uniport_id'].str.replace('up:','')

uniprot_id_df

,Uniport_id,Gene
0,G2Q701,MYCTH_100011
1,G2Q913,MYCTH_100068
2,G2Q8Y3,MYCTH_100089
3,G2Q8N0,MYCTH_100127
4,G2Q7I4,MYCTH_100290
...,...,...
9092,G2Q4M3,MYCTH_99786
9093,G2Q4Q7,MYCTH_99814
9094,G2Q541,MYCTH_99838
9095,G2Q560,MYCTH_99853


In [49]:
"""
合并location, NCBI_gene_id, protein_id, uniprot_id信息
"""

df_gene_info = pd.merge(location_df, ncbi_gene_id_df, on='Gene', how='outer')
df_gene_info = pd.merge(df_gene_info, protein_id_df, on='Gene', how='outer')
df_gene_info = pd.merge(df_gene_info, uniprot_id_df, on='Gene', how='outer')

# 检测是否有重复行
# df_gene_info[df_gene_info.duplicated(['Gene'], keep=False)]

df_gene_info

,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id
0,MYCTH_2114025,NC_016472.1,1381,2142,+,762,11512903,XP_003658314.1,G2Q6N9
1,MYCTH_2293935,NC_016472.1,2744,3343,-,600,11506009,XP_003658315.1,G2Q6P0
2,MYCTH_2293936,NC_016472.1,3344,3817,-,474,11508241,XP_003658316.1,G2Q6P1
3,MYCTH_2051335,NC_016472.1,5330,6385,-,1056,11513239,XP_003658317.1,G2Q6P2
4,MYCTH_2121898,NC_016472.1,15458,25533,+,10076,11511624,XP_003658318.1,G2Q6P3
...,...,...,...,...,...,...,...,...,...
9287,MYCTH_103797,NC_016478.1,4089438,4091337,+,1900,11510115,XP_003667408.1,G2QP41
9288,MYCTH_2071098,NC_016478.1,4095568,4097284,+,1717,11510116,XP_003667409.1,G2QP42
9289,MYCTH_2114022,NC_016478.1,4098122,4098907,-,786,11510117,XP_003667410.1,G2QP43
9290,MYCTH_2114023,NC_016478.1,4100606,4101293,-,688,11510118,XP_003667411.1,G2QP44


In [50]:
# Go.csv存在一对多情况
go_df = pd.read_csv('Go.csv')
go_df.rename(columns={'GOID':'GO', 'DEFINITION':'GO_definition', 'ONTOLOGY':'GO_type', 'TERM':'ONTOLOGY'}, inplace=True)
go_df = go_df.fillna('')

go_df = go_df.groupby('Gene').agg({
    'GO': lambda x: ','.join(x),
    'GO_definition': lambda x: '/'.join(x),
    'ONTOLOGY': lambda x: ','.join(x),
    'GO_type': lambda x: ','.join(x)
}).reset_index()

# 检测是否有重复行
# go_df[go_df.duplicated(['Gene'], keep=False)]

go_df

,Gene,GO,GO_definition,ONTOLOGY,GO_type
0,MYCTH_100011,GO:0006281,The process of restoring DNA after damage. Gen...,DNA repair,BP
1,MYCTH_100068,"GO:0005576,GO:0004553,GO:0005975,GO:0030248",The space external to the outermost structure ...,"extracellular region,hydrolase activity, hydro...","CC,MF,BP,MF"
2,MYCTH_100089,"GO:0005506,GO:0016702",Interacting selectively and non-covalently wit...,"iron ion binding,oxidoreductase activity, acti...","MF,MF"
3,MYCTH_100290,"GO:0000776,GO:0034508",A multisubunit complex that is located at the ...,"kinetochore,centromere complex assembly","CC,BP"
4,MYCTH_100401,"GO:0016020,GO:0015205,GO:0022857,GO:0055085",A lipid bilayer along with all the proteins an...,"membrane,nucleobase transmembrane transporter ...","CC,MF,MF,BP"
...,...,...,...,...,...
5024,MYCTH_99686,GO:0005515,Interacting selectively and non-covalently wit...,protein binding,MF
5025,MYCTH_99786,"GO:0004553,GO:0005975",Catalysis of the hydrolysis of any O-glycosyl ...,"hydrolase activity, hydrolyzing O-glycosyl com...","MF,BP"
5026,MYCTH_99814,"GO:0022857,GO:0055085","Enables the transfer of a substance, usually a...","transmembrane transporter activity,transmembra...","MF,BP"
5027,MYCTH_99838,"GO:0009058,GO:0016747",The chemical reactions and pathways resulting ...,"biosynthetic process,transferase activity, tra...","BP,MF"


In [51]:
kegg_definition = pd.read_csv('KEGG_definition.csv')
kegg_definition.rename(columns={'Pathway':'KEGG_definition'}, inplace=True)

kegg_df = pd.read_csv('KEGG.csv', names=['KEGG', 'KEGG_definition'])

# 合并kegg_definition，kegg_df
kegg_merge_df = pd.merge(kegg_definition, kegg_df, on='KEGG_definition', how='inner')
kegg_merge_df = kegg_merge_df.fillna('')

kegg_merge_df = kegg_merge_df.groupby("Gene").agg({
    'KEGG_definition': lambda x: ','.join(x),
    'KEGG': lambda x: ','.join(x)
}).reset_index()

# 检测是否有重复行
# kegg_merge_df[kegg_merge_df.duplicated(['Gene'], keep=False)]

kegg_merge_df

,Gene,KEGG_definition,KEGG
0,MYCTH_100519,"Inositol phosphate metabolism,Metabolic pathwa...","mtm00562,mtm01100,mtm04070"
1,MYCTH_100654,"Autophagy - yeast,Protein processing in endopl...","mtm04138,mtm04141"
2,MYCTH_101224,"Fatty acid degradation,Tryptophan metabolism","mtm00071,mtm00380"
3,MYCTH_101411,Aminoacyl-tRNA biosynthesis,mtm00970
4,MYCTH_101588,"Inositol phosphate metabolism,Metabolic pathwa...","mtm00562,mtm01100,mtm04070,mtm04138"
...,...,...,...
2091,MYCTH_t95,Aminoacyl-tRNA biosynthesis,mtm00970
2092,MYCTH_t96,Aminoacyl-tRNA biosynthesis,mtm00970
2093,MYCTH_t97,Aminoacyl-tRNA biosynthesis,mtm00970
2094,MYCTH_t98,Aminoacyl-tRNA biosynthesis,mtm00970


In [52]:
"""
合并go_df, kegg_merge_df
"""

df_gene_info = pd.merge(df_gene_info, go_df, on='Gene', how='outer')
df_gene_info = pd.merge(df_gene_info, kegg_merge_df, on='Gene', how='outer')


# 删除Chr这一列值为NaN的行
df_gene_info = df_gene_info.dropna(subset=['Chr'])

df_gene_info

,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id,GO,GO_definition,ONTOLOGY,GO_type,KEGG_definition,KEGG
0,MYCTH_2114025,NC_016472.1,1381.0,2142.0,+,762.0,11512903,XP_003658314.1,G2Q6N9,NaN,NaN,NaN,NaN,NaN,NaN
1,MYCTH_2293935,NC_016472.1,2744.0,3343.0,-,600.0,11506009,XP_003658315.1,G2Q6P0,NaN,NaN,NaN,NaN,NaN,NaN
2,MYCTH_2293936,NC_016472.1,3344.0,3817.0,-,474.0,11508241,XP_003658316.1,G2Q6P1,NaN,NaN,NaN,NaN,NaN,NaN
3,MYCTH_2051335,NC_016472.1,5330.0,6385.0,-,1056.0,11513239,XP_003658317.1,G2Q6P2,GO:0005515,Interacting selectively and non-covalently wit...,protein binding,MF,NaN,NaN
4,MYCTH_2121898,NC_016472.1,15458.0,25533.0,+,10076.0,11511624,XP_003658318.1,G2Q6P3,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9287,MYCTH_103797,NC_016478.1,4089438.0,4091337.0,+,1900.0,11510115,XP_003667408.1,G2QP41,"GO:0016020,GO:0016021,GO:0022857,GO:0055085",A lipid bilayer along with all the proteins an...,"membrane,integral component of membrane,transm...","CC,CC,MF,BP",NaN,NaN
9288,MYCTH_2071098,NC_016478.1,4095568.0,4097284.0,+,1717.0,11510116,XP_003667409.1,G2QP42,NaN,NaN,NaN,NaN,NaN,NaN
9289,MYCTH_2114022,NC_016478.1,4098122.0,4098907.0,-,786.0,11510117,XP_003667410.1,G2QP43,NaN,NaN,NaN,NaN,NaN,NaN
9290,MYCTH_2114023,NC_016478.1,4100606.0,4101293.0,-,688.0,11510118,XP_003667411.1,G2QP44,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
"""
KO id 信息
"""

kunm_df = pd.read_csv('Knum.csv', names=['Protein_id', 'Knum'])
kunm_df = kunm_df.groupby("Protein_id")["Knum"].apply(lambda x: ','.join(x)).reset_index()

kunm_df

,Protein_id,Knum
0,XP_003658319.1,"K01078,K22390"
1,XP_003658326.1,"K07879,K07880"
2,XP_003658327.1,K01192
3,XP_003658328.1,"K00008,K05351"
4,XP_003658329.1,"K01930,K17785"
...,...,...
3838,XP_003667402.1,K14455
3839,XP_003667403.1,K00002
3840,XP_003667406.1,K19355
3841,XP_003667407.1,K00516


In [54]:
"""
EC number
"""

ec_number_df = pd.read_csv('EC_number.csv', names=['Protein_id', 'EC_number'])
ec_number_df = ec_number_df.groupby("Protein_id")["EC_number"].apply(lambda x: ','.join(x)).reset_index()

ec_number_df

,Protein_id,EC_number
0,XP_003658319.1,3.1.3.2
1,XP_003658327.1,3.2.1.25
2,XP_003658328.1,"1.1.1.14,1.1.1.9"
3,XP_003658329.1,6.3.2.17
4,XP_003658330.1,6.3.2.17
...,...,...
1710,XP_003667399.1,4.2.1.20
1711,XP_003667402.1,2.6.1.1
1712,XP_003667403.1,1.1.1.2
1713,XP_003667406.1,3.2.1.78


In [55]:
"""
功能描述信息
"""

description_df = pd.read_csv('Description.csv', names=['Protein_id', 'Description'])

description_df

,Protein_id,Description
0,XP_003658314.1,-
1,XP_003658317.1,phosphoglycerate dehydrogenase activity
2,XP_003658319.1,"Purple acid Phosphatase, N-terminal domain"
3,XP_003658320.1,Integral membrane protein TmpA
4,XP_003658321.1,-
...,...,...
8444,XP_003667408.1,Belongs to the major facilitator superfamily. ...
8445,XP_003667409.1,-
8446,XP_003667410.1,-
8447,XP_003667411.1,Heterokaryon incompatibility protein (HET)


In [56]:
pfam_df = pd.read_csv('PFAM.csv', names=['Protein_id', 'Pfam'])
pfam_df = pfam_df.groupby("Protein_id")["Pfam"].apply(lambda x: ','.join(x)).reset_index()

pfam_df

,Protein_id,Pfam
0,XP_003658314.1,Abhydrolase_6
1,XP_003658319.1,"Metallophos,Metallophos_C,Pur_ac_phosph_N"
2,XP_003658322.1,RNase_T
3,XP_003658323.1,"2-Hacid_dh,2-Hacid_dh_C"
4,XP_003658325.1,adh_short
...,...,...
6732,XP_003667405.1,"BBE,FAD_binding_4,MFS_1"
6733,XP_003667406.1,"CBM_1,Cellulase"
6734,XP_003667407.1,"CBM_20,LPMO_10"
6735,XP_003667408.1,Sugar_tr


In [57]:
"""
利用ProCAST对mt的蛋白序列文件进行注释后，得到Cazy表
"""

cazy_df = pd.read_csv('./Cazy.csv')
cazy_df = cazy_df.iloc[:, :2]

# 去掉无关数据
cazy_df['subject_id'] = cazy_df['subject_id'].str.replace(r'\w+\.1', '', regex=True)

# 去掉多余空格
cazy_df['subject_id'] = cazy_df['subject_id'].str.lstrip()

# 将cazy_df第一列列名改为Protein_id，第二列列名改为Cazy
cazy_df.rename(columns={'query_id':'Protein_id', 'subject_id':'Cazy'}, inplace=True)

# 将第一列的Protein_id相同的行合并，Cazy列的值用逗号隔开
cazy_df = cazy_df.groupby("Protein_id")["Cazy"].apply(lambda x: ','.join(x)).reset_index()

display(cazy_df)

,Protein_id,Cazy
0,XP_003658326.1,GH36: Glycoside Hydrolases (GHs)
1,XP_003658327.1,GH2: Glycoside Hydrolases (GHs)
2,XP_003658329.1,GT4: GlycosylTransferases (GTs)
3,XP_003658332.1,GH36: Glycoside Hydrolases (GHs)
4,XP_003658346.1,GH36: Glycoside Hydrolases (GHs) : Auxiliary A...
...,...,...
765,XP_003667369.1,GH27: Glycoside Hydrolases (GHs)
766,XP_003667383.1,GH47: Glycoside Hydrolases (GHs)
767,XP_003667394.1,CBM20: Carbohydrate-Binding Modules (CBMs)
768,XP_003667406.1,GH5_7: Glycoside Hydrolases (GHs)


In [58]:
"""
合并kunm_df, ec_number_df, description_df, pfam_df, cazy_df
"""

df_gene_info = pd.merge(df_gene_info, kunm_df, on='Protein_id', how='outer')
df_gene_info = pd.merge(df_gene_info, ec_number_df, on='Protein_id', how='outer')
df_gene_info = pd.merge(df_gene_info, description_df, on='Protein_id', how='outer')
df_gene_info = pd.merge(df_gene_info, pfam_df, on='Protein_id', how='outer')
df_gene_info = pd.merge(df_gene_info, cazy_df, on='Protein_id', how='outer')

df_gene_info

,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id,GO,GO_definition,ONTOLOGY,GO_type,KEGG_definition,KEGG,Knum,EC_number,Description,Pfam,Cazy
0,MYCTH_2114025,NC_016472.1,1381.0,2142.0,+,762.0,11512903,XP_003658314.1,G2Q6N9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,Abhydrolase_6,NaN
1,MYCTH_2293935,NC_016472.1,2744.0,3343.0,-,600.0,11506009,XP_003658315.1,G2Q6P0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MYCTH_2293936,NC_016472.1,3344.0,3817.0,-,474.0,11508241,XP_003658316.1,G2Q6P1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MYCTH_2051335,NC_016472.1,5330.0,6385.0,-,1056.0,11513239,XP_003658317.1,G2Q6P2,GO:0005515,Interacting selectively and non-covalently wit...,protein binding,MF,NaN,NaN,NaN,NaN,phosphoglycerate dehydrogenase activity,NaN,NaN
4,MYCTH_2121898,NC_016472.1,15458.0,25533.0,+,10076.0,11511624,XP_003658318.1,G2Q6P3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9287,MYCTH_103797,NC_016478.1,4089438.0,4091337.0,+,1900.0,11510115,XP_003667408.1,G2QP41,"GO:0016020,GO:0016021,GO:0022857,GO:0055085",A lipid bilayer along with all the proteins an...,"membrane,integral component of membrane,transm...","CC,CC,MF,BP",NaN,NaN,K08141,NaN,Belongs to the major facilitator superfamily. ...,Sugar_tr,NaN
9288,MYCTH_2071098,NC_016478.1,4095568.0,4097284.0,+,1717.0,11510116,XP_003667409.1,G2QP42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN
9289,MYCTH_2114022,NC_016478.1,4098122.0,4098907.0,-,786.0,11510117,XP_003667410.1,G2QP43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN
9290,MYCTH_2114023,NC_016478.1,4100606.0,4101293.0,-,688.0,11510118,XP_003667411.1,G2QP44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Heterokaryon incompatibility protein (HET),HET,NaN


In [59]:
nucleic_acid_df = pd.read_csv('Nucleic_acid_sequence.csv')
nucleic_acid_df.rename(columns={'gene':'Gene','sequence':'Nucleic_acid_sequence'}, inplace=True)

nucleic_acid_df

,Gene,Nucleic_acid_sequence
0,MYCTH_2114025,ATGGACCGCCTAAAAGACATCCTGCTCTACATTTGGAACTTCCGCT...
1,MYCTH_2293935,ATGGCCATGACCCGGCAGTCGCCTAGCCAGAGGCTAGTCGCGCCGA...
2,MYCTH_2293936,ATGGACAAAGTGACAAAGCTCCTAGCCAGCAACCCGAGCACAACCG...
3,MYCTH_2051335,ATGGCTGCCTTTCTGCCTCGTGGATATCGCTCCGTCACCGAATACC...
4,MYCTH_2121898,ATGTCGATCGGCACCCTGTCGCCCGCCTCCTACACCCGCTCGAGCG...
...,...,...
9092,MYCTH_103797,ATGCTCGGGATCGAAGCGACGAGAGAGATAACGGCCGCCGAACACG...
9093,MYCTH_2071098,GTCTTTGGGAACGGTGATTCGACGTCTTACCGAGGCCACACCCGAC...
9094,MYCTH_2114022,ATGCCCCTCCTGTGTGAATGTCGCGTAAAGCACGTAACCACGCGAC...
9095,MYCTH_2114023,ATGCGTCTCCTCCACACCACCAAGCTCCACCTCGTCGAGTCCCACA...


In [60]:
amino_acid_sequence_df = pd.read_csv('Amino_acid_sequence.csv')
amino_acid_sequence_df.rename(columns={'Protein ID':'Protein_id','Sequence':'Amino_acid_sequence'}, inplace=True)

amino_acid_sequence_df

,Protein_id,Amino_acid_sequence
0,XP_003658314.1,MDRLKDILLYIWNFRYLPLKYRWSLIKWRRRLLPTRSGLGLRNALR...
1,XP_003658315.1,MAMTRQSPSQRLVAPIRLISIGGALAVRVCHNREASYDIDCMLDPN...
2,XP_003658316.1,MDKVTKLLASNPSTTAYVAGVAVLVLAPQIVAVPALGAIGFGAKGV...
3,XP_003658317.1,MAAFLPRGYRSVTEYRFREDQTDAIVRAAAYHRKDYCRSVIWFSPR...
4,XP_003658318.1,MSIGTLSPASYTRSSASTRPAPVPINFGRSYTRRSSSPVERIARLV...
...,...,...
9092,XP_003667408.1,MLGIEATREITAAEHALGFWPAIQQYPKATLWSMFFCLAVVMAGYD...
9093,XP_003667409.1,VFGNGDSTSYRGHTRPFAFPQPGHTITRVDGRVLCRYPDCRKCAAS...
9094,XP_003667410.1,MPLLCECRVKHVTTRLWEYIFNHVIFTDDKWVVSSQQPPTHQPGEL...
9095,XP_003667411.1,MRLLHTTKLHLVESHSQGILPSAIPPYAILYHTWAKEEGYEKVAGA...


In [61]:
"""
合并nucleic_acid_df, amino_acid_sequence_df
"""

df_gene_info = pd.merge(df_gene_info, nucleic_acid_df, on='Gene', how='outer')
df_gene_info = pd.merge(df_gene_info, amino_acid_sequence_df, on='Protein_id', how='outer')

df_gene_info

,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id,GO,...,GO_type,KEGG_definition,KEGG,Knum,EC_number,Description,Pfam,Cazy,Nucleic_acid_sequence,Amino_acid_sequence
0,MYCTH_2114025,NC_016472.1,1381.0,2142.0,+,762.0,11512903,XP_003658314.1,G2Q6N9,NaN,...,NaN,NaN,NaN,NaN,NaN,-,Abhydrolase_6,NaN,ATGGACCGCCTAAAAGACATCCTGCTCTACATTTGGAACTTCCGCT...,MDRLKDILLYIWNFRYLPLKYRWSLIKWRRRLLPTRSGLGLRNALR...
1,MYCTH_2293935,NC_016472.1,2744.0,3343.0,-,600.0,11506009,XP_003658315.1,G2Q6P0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ATGGCCATGACCCGGCAGTCGCCTAGCCAGAGGCTAGTCGCGCCGA...,MAMTRQSPSQRLVAPIRLISIGGALAVRVCHNREASYDIDCMLDPN...
2,MYCTH_2293936,NC_016472.1,3344.0,3817.0,-,474.0,11508241,XP_003658316.1,G2Q6P1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ATGGACAAAGTGACAAAGCTCCTAGCCAGCAACCCGAGCACAACCG...,MDKVTKLLASNPSTTAYVAGVAVLVLAPQIVAVPALGAIGFGAKGV...
3,MYCTH_2051335,NC_016472.1,5330.0,6385.0,-,1056.0,11513239,XP_003658317.1,G2Q6P2,GO:0005515,...,MF,NaN,NaN,NaN,NaN,phosphoglycerate dehydrogenase activity,NaN,NaN,ATGGCTGCCTTTCTGCCTCGTGGATATCGCTCCGTCACCGAATACC...,MAAFLPRGYRSVTEYRFREDQTDAIVRAAAYHRKDYCRSVIWFSPR...
4,MYCTH_2121898,NC_016472.1,15458.0,25533.0,+,10076.0,11511624,XP_003658318.1,G2Q6P3,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ATGTCGATCGGCACCCTGTCGCCCGCCTCCTACACCCGCTCGAGCG...,MSIGTLSPASYTRSSASTRPAPVPINFGRSYTRRSSSPVERIARLV...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9287,MYCTH_103797,NC_016478.1,4089438.0,4091337.0,+,1900.0,11510115,XP_003667408.1,G2QP41,"GO:0016020,GO:0016021,GO:0022857,GO:0055085",...,"CC,CC,MF,BP",NaN,NaN,K08141,NaN,Belongs to the major facilitator superfamily. ...,Sugar_tr,NaN,ATGCTCGGGATCGAAGCGACGAGAGAGATAACGGCCGCCGAACACG...,MLGIEATREITAAEHALGFWPAIQQYPKATLWSMFFCLAVVMAGYD...
9288,MYCTH_2071098,NC_016478.1,4095568.0,4097284.0,+,1717.0,11510116,XP_003667409.1,G2QP42,NaN,...,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,GTCTTTGGGAACGGTGATTCGACGTCTTACCGAGGCCACACCCGAC...,VFGNGDSTSYRGHTRPFAFPQPGHTITRVDGRVLCRYPDCRKCAAS...
9289,MYCTH_2114022,NC_016478.1,4098122.0,4098907.0,-,786.0,11510117,XP_003667410.1,G2QP43,NaN,...,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,ATGCCCCTCCTGTGTGAATGTCGCGTAAAGCACGTAACCACGCGAC...,MPLLCECRVKHVTTRLWEYIFNHVIFTDDKWVVSSQQPPTHQPGEL...
9290,MYCTH_2114023,NC_016478.1,4100606.0,4101293.0,-,688.0,11510118,XP_003667411.1,G2QP44,NaN,...,NaN,NaN,NaN,NaN,NaN,Heterokaryon incompatibility protein (HET),HET,NaN,ATGCGTCTCCTCCACACCACCAAGCTCCACCTCGTCGAGTCCCACA...,MRLLHTTKLHLVESHSQGILPSAIPPYAILYHTWAKEEGYEKVAGA...


In [63]:
# 保存文件
df_gene_info.to_csv('gene_information.csv', index=False)

In [70]:
df_change = pd.read_csv('gene_information.csv')

df_dict = df_change.fillna('null').to_dict(orient='records')

In [71]:
output_dict = {}

for item in df_dict:
    Gene = 'KEY:' + item['Gene']
    output_dict[Gene] = {
        'Gene': item['Gene'],
        'NCBI_gene_id': int(item['NCBI_gene_id']) if item['NCBI_gene_id'] != 'null' else None,
        'Protein_id': item['Protein_id'],
        'Uniport_id': item['Uniport_id'],
        'Knum': str(item['Knum']),
        'Description': item['Description'],
        'Cazy': item['Cazy'],
        'PFAM': item['Pfam'],
        'Chr': item['Chr'],
        'Start': int(item['Start']) if item['Start'] != 'null' else None,
        'End': int(item['End']) if item['End'] != 'null' else None,
        'Strand': item['Strand'],
        'Length': int(item['Length']) if item['Length'] != 'null' else None,
        'Nucleic_acid_sequence': item['Nucleic_acid_sequence'],
        'Amino_acid_sequence': item['Amino_acid_sequence'],
        'GO_information': {
            'GO': item['GO'],
            'GO_definition': item['GO_definition'],
            'ONTOLOGY': item['ONTOLOGY'],
            'GO_type': item['GO_type'],
        },
        'KEGG_information': {
            'KEGG': str(item['KEGG']),
            'KEGG_definition': str(item['KEGG_definition']),
        },
        'EC_number': str(item['EC_number']),
    }

In [72]:
for key, value in output_dict.items():
    GO = value['GO_information']['GO'].split(',')
    GO_definition = value['GO_information']['GO_definition'].split('/')
    ONTOLOGY = value['GO_information']['ONTOLOGY'].split(',')
    GO_type = value['GO_information']['GO_type'].split(',')
    GO_information = []
    for i in range(len(GO)):
        GO_information.append({
            'GO': GO[i],
            'GO_definition': GO_definition[i],
            'ONTOLOGY': ONTOLOGY[i],
            'GO_type': GO_type[i]
        })
    output_dict[key]['GO_information'] = GO_information

for key, value in output_dict.items():
    KEGG = value['KEGG_information']['KEGG'].split(',')
    KEGG_definition = value['KEGG_information']['KEGG_definition'].split(',')
    KEGG_information = []
    for i in range(len(KEGG)):
        KEGG_information.append({
            'KEGG': KEGG[i],
            'KEGG_definition': KEGG_definition[i],
        })
    output_dict[key]['KEGG_information'] = KEGG_information

output_dict

{'KEY:MYCTH_2114025': {'Gene': 'MYCTH_2114025',
  'NCBI_gene_id': 11512903,
  'Protein_id': 'XP_003658314.1',
  'Uniport_id': 'G2Q6N9',
  'Knum': 'null',
  'Description': '-',
  'Cazy': 'null',
  'PFAM': 'Abhydrolase_6',
  'Chr': 'NC_016472.1',
  'Start': 1381,
  'End': 2142,
  'Strand': '+',
  'Length': 762,
  'Nucleic_acid_sequence': 'ATGGACCGCCTAAAAGACATCCTGCTCTACATTTGGAACTTCCGCTATCTCCCTCTCAAATACCGGTGGTCTCTCATCAAATGGAGGCGCCGCCTCTTGCCGACGAGGAGTGGGCTCGGACTTCGAAACGCACTGCGCAATGCCCGCTCACTCAAGAACCCGCCAGTACCACACCCTTACCTGTTCGTCCTCCGGCTCCTCTGGCCCTTCCCAACTTGGTACTACCCGGCCGTGCTCCCGCCACCGCCGCGGGAAATCATGGCCAACCCACGCAGGGTCAGCTGCCAGGCCCGCGACCTGATCTATCTCCGCTCAATGCCGCTCTGGCGGGCGCGCGACACGCCACAACGGTCGTTTTATCGTCTATACGAGGCCCTTTGCGCCGCGGACGAGCACATGATTACCTATGAAACCGAGTACTTCTGGCGCCAATCATCACCGCGCTGGGCCACGGCCAACATCCCGGACCCCCAGTGCGAAGACCTAGAGCAGTACGCTGTCATGGCCTCGTTGGCCGAGGTCTTGGTCAAGTCGTTCATGTGGCGTCTCGAACTCGGTCTGCGGCGCCATGATACAGCGATCATGAACCGCAACGTCGACCCTCCCCCGTTTGTGCCGGAGGTTTGCCCTTCGTGGACAGCCAAGGTGCCGCCATTGACGG

In [73]:
with open('gene_information.json', 'w') as f:
    json.dump(output_dict, f, indent=4)